<div dir="rtl">

# 02 — סטטיסטיקות ליגה

**פעימה A: הקבוצה בעונה ובין עונות.** השאלה: מה מבדיל אלופה מיורדת, כמה עונה אחת קשורה לבאה וכמה הכסף מסביר?

זו מחברת תיאורית: הנתונים כוללים את סוף העונה. אין כאן פיצ'רים, מודלים, כיוונון או כתיבה לדאטה הנקי. סיכום של עונה אינו קלט למודל באותה עונה; בעתיד מותר להעביר מידע מעונה קודמת דרך `prematch` בלבד.

כל החישובים מוגבלים ל-development: עונות 8–25 (2006/07–2023/24). עונות ההחזקה והעונה החיה אינן נטענות או מוצגות.

## תוכן עניינים — חמש פעימות

1. **A — הקבוצה בעונה ובין עונות** — בפרק הנוכחי.
2. **B — הליגה לאורך השנים** — ספי נקודות, ביתיות, עידנים ולוח השנה.
3. **C — מהלך העונה והטבלה** — התייצבות, פיצול, פלייאוף ומחזורים אחרונים.
4. **D — המשחק והשערים** — תוצאות, Poisson תיאורי, מחצית, ציר זמן ואדומים.
5. E — שופטים, מאמנים, אצטדיונים ומשחקים גדולים — טרם מומש.

</div>

<div dir="rtl">

### סביבת התצוגה
השאלה: באילו עונות וקבוצה נתמקד בדוגמאות? מגדירים את 2023/24 ואת מכבי חיפה, ואת אותם צבעי בית/תיקו/חוץ של מחברת 01. כל חישוב מיובא מהחבילה.

</div>

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.io as pio
from IPython.display import display

from ligat.analysis.league_stats import (
    persistence_summary,
    position_profile,
    promoted_value_check,
    promoted_value_summary,
    season_persistence,
    team_consistency,
    team_season_stats,
    team_style_profile,
    value_summary,
    value_vs_performance,
)
from ligat.analysis.prepare import DEVELOPMENT_SEASONS, load_analysis_tables
from ligat.data.repository import Repository
from ligat.viz.league_stats import AWAY_COLOR, DRAW_COLOR, HOME_COLOR, chapter_a_chart

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root)
EXAMPLE_SEASON = 25
TEAM = "מכבי חיפה"
OUTCOME_COLORS = {"בית": HOME_COLOR, "תיקו": DRAW_COLOR, "חוץ": AWAY_COLOR}
pd.set_option("display.max_columns", 15)
pio.renderers.default = "plotly_mimetype+notebook"
display(
    pd.DataFrame(
        {
            "הגדרה": ["עונת דוגמה", "קבוצה", "עונות מחקר"],
            "ערך": ["2023/24", TEAM, "2006/07–2023/24 בלבד"],
        }
    )
)

<div dir="rtl">

### טעינה מוגבלת לפיתוח
השאלה: כמה משחקים ועונות באמת נכללים? קוראים דרך Repository רק את עונות הפיתוח. הטבלאות הרשמיות שנקראות בתשתית הקיימת מספקות תוויות מיקום וירידה תיאוריות בלבד; אין כאן קלט למודל או אימון. [משרת: הבנת הליגה]

</div>

In [ ]:
tables = load_analysis_tables(Repository(), seasons=DEVELOPMENT_SEASONS)
display(
    pd.DataFrame(
        {
            "תצוגה": ["משחקים", "קבוצה × משחק", "קבוצה × עונה"],
            "שורות": [len(tables.match_table), len(tables.team_matches), len(tables.team_seasons)],
        }
    )
)

<div dir="rtl">

## A — הקבוצה בעונה ובין עונות

> ### איך לקרוא את הפרק
> **נקודות מול מיקום:** נקודות למשחק מחושבות ממשחקים ששוחקו, ללא עונשים וללא חציית הנקודות של 2009/10–2010/11; המיקום הסופי רשמי ויכול לשקף גם החלטות מנהלתיות ומשחקים טכניים שהוחרגו.
>
> **n וגודל מדגם:** n הוא מספר התצפיות שהשתתפו בחישוב; מציינים בנפרד גם כמה עונות ומשחקים יש. בקבוצה עם פחות מ-30 תצפיות לא מסיקים מסקנה ולא מדרגים איכות.
>
> **רווחי 95%:** Wilson לשיעורי משחקים; לממוצעים דוגמים עונות, ולמתאמים דוגמים בלוקי עונה עם כל הקבוצות שבאותה עונה. 2,000 דגימות לממוצעים ו-1,000 לקשרים, עם זרע קבוע; הרווחים אינם מבטלים תלות בין עונות ואינם תיקון להשוואות מרובות.
>
> **חוסרים:** אין שווי קודם לעונה הראשונה ולעולות, ויש חוסר במקור. חוסר אינו אפס. סטיית תקן חסרה בפחות משלוש עונות. נתוני מחצית ותזמון שערים אינם משמשים בפרק זה, לכן חריגי המחצית אינם מסננים תוצאות משחק מלא.
>
> **שווי העולות:** ערך הכלל הוא חציון החצי התחתון לפי שווי בעונה הקודמת. העולה אינה חלק מהליגה הקודמת. השוואת אותה עונה ללא העולות היא משנית בלבד; החצי התחתון מחושב מהערכים הזמינים, ובמספר אי-זוגי נלקח floor(n/2).
>
> **קשר אינו סיבה:** שווי סוף עונה וסיכומי עונה נשארים תיאוריים. הרגרסיה מתארת את המדגם, לא בודקת חיזוי עתידי. השוואות רבות עלולות לייצר הבדלים מקריים; כל ממצא צריך בדיקה בהמשך.

### עונת דוגמה: 2023/24
**השאלה:** איך נראית עונת קבוצה במספרים?
**מה לחפש בטבלה:** השוו נקודות, הבקעה וספיגה בין הצמרת לתחתית.
**התשובה הכדורגלנית:** עונה מוצלחת משלבת צבירת נקודות עם הבקעה והגנה; מיקום לבדו אינו מתאר איך הקבוצה שיחקה. [משרת: הבנת הליגה, goals]

</div>

In [ ]:
season_stats = team_season_stats(tables)
display(
    season_stats.xs(EXAMPLE_SEASON, level="season_id")
    .sort_values("final_position")[
        [
            "team_name",
            "final_position",
            "matches_played",
            "wins",
            "draws",
            "losses",
            "points_per_match",
            "goals_scored_per_match",
            "goals_conceded_per_match",
            "home_points_per_match",
            "away_points_per_match",
            "big_win_rate",
            "btts_rate",
            "over_2_5_rate",
            "relegated",
        ]
    ]
    .round(3)
)

<div dir="rtl">

### הקבוצה שבחרנו לאורך עונות הפיתוח
**השאלה:** האם מכבי חיפה מציגה מאפיינים דומים מעונה לעונה?

**מה לחפש בגרף או בטבלה:** הסתכלו על השינויים בנקודות ועל רצפים ללא הפסד או ניצחון; אפשר לשנות TEAM בתא ההגדרות.

**התשובה הכדורגלנית:** אותה קבוצה יכולה להציג עונות שונות מאוד, ולכן נבחן בהמשך כמה להסתמך על העונה הקודמת. [prematch]

</div>

In [ ]:
display(
    season_stats.loc[
        season_stats.team_name.eq(TEAM),
        [
            "team_name",
            "final_position",
            "points_per_match",
            "goals_scored_per_match",
            "goals_conceded_per_match",
            "longest_unbeaten_run",
            "longest_winless_run",
        ],
    ]
    .tail(8)
    .round(3)
)

<div dir="rtl">

### כמה שווה כל מקום — מפרידים עידנים
**השאלה:** באיזה קצב נקודות סיימו הקבוצות בכל מקום?

**מה לחפש בגרף או בטבלה:** השוו את ממוצע הנקודות ואת הטווח בין עונות, בנפרד לכל גודל ליגה.

**התשובה הכדורגלנית:** בעידן 14 הקבוצות האלופה צברה בממוצע כ-2.28 נקודות למשחק — קצב של כ-82 נקודות ב-36 משחקים. [motivation, table_model]

</div>

In [ ]:
profiles = position_profile(tables)
display(
    profiles.xs("14_teams", level="era")[
        [
            "n",
            "points_per_match_mean",
            "points_per_match_median",
            "points_per_match_min",
            "points_per_match_max",
            "points_per_match_ci_lower",
            "points_per_match_ci_upper",
        ]
    ].round(3)
)

<div dir="rtl">

### גרף 1 — התפלגות נקודות לפי מיקום
**השאלה:** כמה רחב טווח הנקודות לכל מקום בעידן 14 הקבוצות?

**מה לחפש בגרף או בטבלה:** כל נקודה היא עונת קבוצה; קופסה גבוהה ורחבה מעידה על פיזור גדול יותר.

**התשובה הכדורגלנית:** אין מספר נקודות אחד שמבטיח אליפות או הישרדות; הסף משתנה בין העונות. [motivation, table_model]

</div>

In [ ]:
chapter_a_chart("position", season_stats).show()

<div dir="rtl">

### גרף 2 — התקפה והגנה יחד
**השאלה:** מה מבדיל בין עונת צמרת לעונת תחתית?

**מה לחפש בגרף או בטבלה:** מימין מבקיעים יותר; נמוך בגרף סופגים פחות. הצבע מציין מיקום רשמי.

**התשובה הכדורגלנית:** קבוצה מצליחה צריכה לשלב התקפה והגנה; שני הצדדים יהיו חשובים בחיזוי שערים. [goals]

</div>

In [ ]:
chapter_a_chart("attack_defense", season_stats).show()

<div dir="rtl">

### גרף 3 — בית מול חוץ
**השאלה:** איפה הקבוצות צברו יותר נקודות?

**מה לחפש בגרף או בטבלה:** מתחת לאלכסון נצברו יותר נקודות בבית, ומעליו יותר בחוץ.

**התשובה הכדורגלנית:** בממוצע עונות־קבוצה הבית הוסיף כ-0.23 נקודות למשחק; זהו קשר תיאורי, לא מדידת השפעה סיבתית. [goals, prematch]

</div>

In [ ]:
chapter_a_chart("home_away", season_stats).show()

<div dir="rtl">

### גרף 4 — ניצחונות גדולים מול צרים
**השאלה:** האם הניצחונות מגיעים בהפרש גדול או קטן?

**מה לחפש בגרף או בטבלה:** הציר האופקי מציג ניצחון בהפרש 3 ומעלה; האנכי ניצחון בהפרש שער אחד.

**התשובה הכדורגלנית:** גם ניצחון דחוק וגם ניצחון גדול מעניקים שלוש נקודות, אך הם מספרים סיפור שונה על המשחק. [goals, scores]

</div>

In [ ]:
chapter_a_chart("win_style", season_stats).show()

<div dir="rtl">

### סגנון משחק על פני לפחות חמש עונות
**השאלה:** באילו קבוצות משחקים עם שערים בשני הצדדים נפוצים יותר?

**מה לחפש בגרף או בטבלה:** השוו את שיעורי שתי הקבוצות מבקיעות ושלושה שערים ומעלה, לצד כמות העונות.

**התשובה הכדורגלנית:** קבוצות יכולות להיות פתוחות או סגורות יותר, בלי שזה כשלעצמו מעיד על איכות. [scores, goals]

</div>

In [ ]:
styles = team_style_profile(tables)
display(
    styles[
        [
            "team_name",
            "n_seasons",
            "n_matches",
            "btts_rate",
            "btts_rate_ci_lower",
            "btts_rate_ci_upper",
            "over_2_5_rate",
            "over_2_5_rate_ci_lower",
            "over_2_5_rate_ci_upper",
        ]
    ]
    .head(10)
    .round(3)
)

<div dir="rtl">

### גרף 5 — קבוצות פתוחות וסגורות
**השאלה:** כמה שונים שיעורי המשחקים הפתוחים בין הקבוצות?

**מה לחפש בגרף או בטבלה:** השוו את הנקודות ואת טווחי אי-הוודאות; טווחים חופפים מחייבים זהירות.

**התשובה הכדורגלנית:** הבדל קטן בשיעור אינו בהכרח דפוס יציב שאפשר להסתמך עליו במשחק הבא. [scores]

</div>

In [ ]:
chapter_a_chart("open_style", styles).show()

<div dir="rtl">

### רצף בין עונות — מי שיחקה גם בעונה הקודמת?
**השאלה:** האם העונה הקודמת קשורה לבאה?

**מה לחפש בגרף או בטבלה:** כל שורה מחברת אותה קבוצה בשתי עונות עוקבות, בלי לדלג מעל עונת היעדרות.

**התשובה הכדורגלנית:** לעולה אין עונה קודמת בליגת העל, ולכן צריך להבחין בינה לבין קבוצה ותיקה. [prematch, table_model]

</div>

In [ ]:
pairs = season_persistence(tables)
display(pairs.head(8).round(3))

<div dir="rtl">

### מתאם ומקדם התמדה
**השאלה:** כמה מהפער בעונה הקודמת נשמר בקשר ההיסטורי?

**מה לחפש בגרף או בטבלה:** שיפוע קרוב ל-1 מציין התמדה חזקה יותר; הסתכלו גם על המתאמים ועל טווח השיפוע.

**התשובה הכדורגלנית:** מקדם 0.77 אומר שפער של חצי נקודה למשחק קשור לכ-0.39 נקודה בעונה הבאה — לא ל-77% דיוק חיזוי. [prematch]

</div>

In [ ]:
persistence = persistence_summary(tables)
display(
    persistence[
        [
            "n",
            "n_seasons",
            "pearson",
            "spearman",
            "slope",
            "intercept",
            "slope_ci_lower",
            "slope_ci_upper",
        ]
    ].round(3)
)

<div dir="rtl">

### גרף 6 — העונה הקודמת מול הנוכחית
**השאלה:** האם הקבוצות שומרות על קצב צבירת הנקודות?

**מה לחפש בגרף או בטבלה:** השוו את הקו הכחול לאלכסון; הפיזור סביבו מראה עד כמה העונות שונות.

**התשובה הכדורגלנית:** העבר מספק כיוון, אך אינו קובע מה יקרה בעונה הבאה. [prematch]

</div>

In [ ]:
chapter_a_chart("persistence", pairs, persistence).show()

<div dir="rtl">

### כסף וביצועים — שני סוגי שווי
**השאלה:** האם שווי קודם קשור לנקודות של העונה הנוכחית?

**מה לחפש בגרף או בטבלה:** השוו בין נתח השווי בעונה הקודמת לבין נתח השווי באותה עונה; NA נשאר חסר.

**התשובה הכדורגלנית:** השווי של העונה הקודמת מתאים למחקר על חיזוי מראש; השווי העונתי הנוכחי כולל גם את המשך העונה. [goals]

</div>

In [ ]:
values = value_vs_performance(tables)
display(values.xs(EXAMPLE_SEASON, level="season_id").head(8).round(3))

<div dir="rtl">

### סיכום קשרי השווי
**השאלה:** כמה חזק הקשר בין שווי, גודל סגל ונקודות?

**מה לחפש בגרף או בטבלה:** ספירמן חיובי מציין סדר דומה בין הקבוצות; ערך שלילי מציין סדר הפוך.

**התשובה הכדורגלנית:** שווי קודם נמצא בקשר חיובי עם נקודות, וגודל הסגל בקשר שלילי חלש; קשר אינו סיבה. [goals]

</div>

In [ ]:
value_relationships = value_summary(tables)
display(
    value_relationships[
        [
            "n",
            "n_seasons",
            "pearson",
            "spearman",
            "spearman_ci_lower",
            "spearman_ci_upper",
            "slope",
            "intercept",
        ]
    ].round(3)
)

<div dir="rtl">

### גרף 7 — חריגות מול שווי קודם
**השאלה:** מי צברה יותר או פחות נקודות מהקשר הממוצע לשווי?

**מה לחפש בגרף או בטבלה:** הסימונים מצביעים על עד עשר עונות מעל הקו ועד עשר מתחתיו.

**התשובה הכדורגלנית:** חריגה יכולה לסמן עונה מפתיעה; היא אינה הוכחה לניהול טוב או רע או להצלחת חיזוי. [goals]

</div>

In [ ]:
chapter_a_chart("value", values, value_relationships).show()

<div dir="rtl">

### עולות וכלל ההשלמה
**השאלה:** האם כלל השווי נותן לעולות ערך דומה לשווי שנצפה?

**מה לחפש בגרף או בטבלה:** השוו את ערך הכלל מעונת t−1 לשווי העולה בעונה t; העמודה same_season_share_excl_promoted מציגה השוואה נפרדת לעונה t ללא העולות.

**התשובה הכדורגלנית:** הכלל מבוסס על קבוצות חלשות יחסית בליגה הקודמת; שווי העולה בעונה הנוכחית משמש רק לבדיקה בדיעבד. [prematch]

</div>

In [ ]:
promoted = promoted_value_check(tables)
display(promoted.head(10).round(4))

<div dir="rtl">

### סיכום בדיקת ההשלמה
**השאלה:** בכמה עולות ערך הכלל גבוה מהשווי שנצפה?

**מה לחפש בגרף או בטבלה:** fraction_below הוא שיעור העולות מתחת לערך הכלל; n_missing מראה כמה נשארו ללא שווי.

**התשובה הכדורגלנית:** הכלל מעריך את העולות מעט גבוה מדי במדגם הזה; זה ממצא לבדיקה, והכלל אינו משתנה כאן. [prematch]

</div>

In [ ]:
promoted_summary = promoted_value_summary(tables)
display(promoted_summary.round(4))

<div dir="rtl">

### גרף 8 — שווי העולה מול ערך הכלל
**השאלה:** כמה העולות רחוקות מהערך לפי הליגה הקודמת?

**מה לחפש בגרף או בטבלה:** מתחת לאלכסון השווי הממשי נמוך מערך הכלל; מעליו הוא גבוה ממנו.

**התשובה הכדורגלנית:** כ-72% מהעולות עם שווי זמין נמצאות מתחת לכלל — ברוב המקרים הוא מעניק להן ערך גבוה יותר מהשווי שנצפה. [prematch]

</div>

In [ ]:
chapter_a_chart("promoted", promoted).show()

<div dir="rtl">

### יציבות ויו-יו בין עונות
**השאלה:** מי מציגה קצב נקודות דומה לאורך השנים, ומי עולה שוב ושוב?

**מה לחפש בגרף או בטבלה:** הסתכלו על הממוצע, התנודתיות וכמות העונות; יו-יו מוגדרת כאן לפחות שתי עליות נצפות.

**התשובה הכדורגלנית:** למועדון עם מעט עונות יש פחות מידע על יציבות; עליות וירידות יכולות להעיד על תנודתיות לאורך זמן. [table_model, הבנת הליגה]

</div>

In [ ]:
consistency = team_consistency(tables)
display(
    consistency[
        [
            "team_name",
            "seasons",
            "mean_ppm",
            "std_ppm",
            "mean_ppm_ci_lower",
            "mean_ppm_ci_upper",
            "best_final_position",
            "worst_final_position",
            "titles",
            "top3_seasons",
            "relegations",
            "promotions",
            "is_yoyo",
        ]
    ]
    .head(10)
    .round(3)
)

<div dir="rtl">

### גרף 9 — יציבות מול ממוצע
**השאלה:** האם ממוצע גבוה מגיע עם תנודתיות קטנה?

**מה לחפש בגרף או בטבלה:** נקודה נמוכה מציינת שינוי קטן יותר בין עונות; גודל הנקודה מציין כמה עונות נצפו.

**התשובה הכדורגלנית:** יציבות וכוח ממוצע הם שני דברים שונים: קבוצה יכולה להיות יציבה גם ברמת נקודות נמוכה. [table_model]

</div>

In [ ]:
chapter_a_chart("consistency", consistency).show()

<div dir="rtl">

## מה למדנו — פעימה A

1. אלופה בעידן 14 הקבוצות צברה בממוצע כ-2.28 נקודות למשחק — קצב של כ-82 נקודות ב-36 משחקים, שנותן הקשר למאבק האליפות. [motivation, table_model]
2. הבית קשור לתוספת של כ-0.23 נקודות למשחק בממוצע עונות־קבוצה, ולכן נבחן ביצועי בית וחוץ בנפרד. [goals, prematch]
3. מקדם ההתמדה הוא כ-0.77: פער של 0.5 נקודה מעל הממוצע קשור לכ-0.39 בעונה הבאה, עם שונות רבה בין הקבוצות — זה קשר היסטורי ולא תחזית. [prematch]
4. שווי קודם נמצא בקשר חיובי עם נקודות, ספירמן כ-0.56 על 202 זוגות מלאים; שווי אותה עונה בקשר חזק יותר, כ-0.62, אך אינו מידע זמין לפני אותה עונה. [goals]
5. גודל הסגל נמצא בקשר שלילי חלש עם נקודות, ספירמן כ-‎−0.19; פירוש אפשרי שלא נבדק כאן הוא שקבוצות במשבר מחליפות יותר שחקנים ודף העונה סופר את כולם — קשר, לא סיבה. [goals, הבנת הליגה]
6. ב-26 מתוך 36 עולות עם שווי זמין, כ-72%, השווי נמוך מערך הכלל מעונת t−1, וחציון ההפרש הוא כ-‎−0.0039 מנתח הליגה; הכלל מעריך את העולות מעט גבוה מדי במדגם הזה, אבל זו בדיקת סבירות בלבד והכלל לא משתנה כאן. [prematch]

פעימות B–E טרם מומשו. נעצור לבדיקה חוזרת ולאישור מיזוג A לפני הפרק הבא.

</div>

<div dir="rtl">

## B — הליגה לאורך השנים

> ### איך לקרוא את הפרק
> **שתי נקודות זמן:** קו העליון נמדד בסוף השלב הסדיר; אליפות והישרדות בסוף העונה. נקודות כוללות אינן תמיד מסודרות בין היעדים: ב־2018/19 וב־2023/24 סף ההישרדות הסופי גבוה מקו העליון המוקדם. גם נקודות למשחק משוות שלבים ויריבים שונים.
>
> **נקודות מול מיקום:** כל הנקודות הן 3 לניצחון ו־1 לתיקו, בלי עונשים ובלי חצייה. האלופה, הסגנית והיורדות מסומנות לפי התוצאה הרשמית. ב־2009/10 האלופה הרשמית צברה 84 נקודות ספורטיביות לעומת 87 לסגנית; ב־2016/17 יורדת צברה 38 לעומת שורדת עם 32. ב־2009/10 החצייה הכריעה את האליפות; ב־2016/17 הפועל ת״א ירדה בעקבות הורדה של 9 נקודות, למרות 38 נקודות על המגרש לעומת שורדת עם 32. אלה סיפורים כדורגלניים של חוקים ועונשים, לא באגים בחישוב הספורטיבי. בודקים את שוברי השוויון ואת חוקי הליגה במחברת 03.
>
> **מספר משחקים:** העליון בדרך כלל 36 והתחתון 33 בעידן 14 הקבוצות; ההחרגות הטכניות עשויות להפחית משחקים שנצפים כאן. ליד כל סף נשמר גם מספר המשחקים. בשוויון ניקוד משתמשים בממוצע מספר המשחקים של הקבוצות השוות לצורך שיעור תיאורי בלבד.
>
> **אי־ודאות:** n הוא מספר המשחקים או העונות שהשתתפו בחישוב. שיעורים מוצגים עם רווחי Wilson של 95%; ממוצעי שערים לפי יום/שעה/עידן עם bootstrap של עונות, 2,000 דגימות וזרע קבוע. רווחים אלה אינם תיקון להשוואות מרובות. לא מסיקים מסקנה מקבוצה עם פחות מ־30 משחקים. הטבלאות תיאוריות, וקשר אינו סיבה.
>
> **כיסוי ומגבלות:** 4,315 משחקי פיתוח בלבד, ללא 11 המשחקים הטכניים שכבר הוחרגו. אין כתיבה לנתונים הנקיים. שלוש היורדות ב־2011/12 נקראות מהדאטה; הקונפיגורציה הישנה לא מתוקנת כאן. שערי פנדל אינם פנדלים שנשרקו, ואדומים אינם כלל הכרטיסים.

**השאלה:** מאילו טבלאות ומאיזה פורמט נגזור את היעדים?

**מה לחפש:** גודל העליון הוא 6 או 8 לפי העונה; אין קו עליון בעונות ללא פיצול.

**התשובה הכדורגלנית:** ב־14 מעונות הפיתוח הכניסה לעליון היא לשישייה, בעונה אחת לשמינייה וב־3 אין פיצול; אותו מקום מספרי אינו אותו יעד בכל פורמט.

</div>

In [ ]:
from ligat.analysis.league_stats import (
    calendar_effects,
    champions_summary,
    competitive_balance,
    critical_ties,
    critical_ties_summary,
    era_comparison,
    home_advantage_by_season,
    home_advantage_periods,
    postponed_summary,
    season_thresholds,
    threshold_ranges,
)
from ligat.analysis.prepare import load_playoff_upper_sizes
from ligat.viz.league_history import chapter_b_chart

upper_sizes = load_playoff_upper_sizes()
display(pd.DataFrame({"גודל הפלייאוף העליון": upper_sizes}).loc[list(DEVELOPMENT_SEASONS)].head(8))

<div dir="rtl">

### כמה נקודות נצברו לכל יעד?
**השאלה:** באיזה קצב צברו נקודות האלופה, השורדת והקבוצה שעל קו העליון?

**מה לחפש בטבלה:** כל שורה היא עונה; NA בקו העליון מציין שלא היה פיצול לפלייאוף.

**התשובה הכדורגלנית:** האלופה צברה כ־2.0–2.5 נקודות למשחק, והשורדת הנמוכה סביב 1.0; הפער נותן קנה מידה למרוץ אליפות מול מאבק הישרדות, לא הבטחה לניקוד מספיק.

</div>

In [ ]:
thresholds = season_thresholds(tables, upper_sizes=upper_sizes)
display(
    thresholds[
        [
            "champion_ppm",
            "upper_line_ppm",
            "safety_ppm",
            "relegated_max_ppm",
            "games_per_team_min",
            "games_per_team_max",
            "n_relegated",
            "flag_inverted",
        ]
    ]
    .head(8)
    .round(3)
)

<div dir="rtl">

### גרף B1 — ספי הנקודות לאורך העונות
**השאלה:** האם הקצב הנדרש לאליפות ולהישרדות נשאר דומה?

**מה לחפש בגרף:** קו האלופה גבוה בדרך כלל, אך קו ההישרדות עשוי לחצות את קו היורדת. קו העליון מתייחס לשלב מוקדם יותר.

**התשובה הכדורגלנית:** קצב האלופה נע בין 1.97 ל־2.49 נקודות למשחק, לעומת 0.85–1.16 אצל השורדת הנמוכה; יעד הנקודות לחשיבות משחק צריך להתחשב בעונה ובשלב. [motivation, table_model]

</div>

In [ ]:
chapter_b_chart("thresholds", thresholds).show()

<div dir="rtl">

### בעידן 14 הקבוצות — כמה נקודות בסך הכול?
**השאלה:** מה הטווח שנצפה לניקוד של הקבוצות בכל יעד?

**מה לחפש בטבלה:** השוו חציון וטווח, ובדקו את מספר המשחקים: העליון נמדד לאחר 25–26 משחקים שנכללו, ההישרדות לאחר 32–33.

**התשובה הכדורגלנית:** השורדת בעלת הניקוד הנמוך צברה בדרך כלל סביב 34 נקודות, בטווח 28–37; אין בכך הבטחה להישארות.

</div>

In [ ]:
ranges_14 = threshold_ranges(tables, upper_sizes=upper_sizes)
display(ranges_14.round(3))

<div dir="rtl">

### גרף B2 — טווחי היעדים
**השאלה:** כמה שונות העונות זו מזו באותו פורמט?

**מה לחפש בגרף:** כל נקודה היא עונה; התיבה מציגה את מרכז ההתפלגות. בריחוף מופיעים העונה ומספר המשחקים.

**התשובה הכדורגלנית:** חציון האלופה הוא 83.5 נקודות, אבל אין להשוות אותו ישירות לסף עליון אחרי השלב הסדיר.

</div>

In [ ]:
chapter_b_chart("threshold_boxes", thresholds).show()

<div dir="rtl">

### שוויון על הקו
**השאלה:** בכמה עונות היה שוויון נקודות בין קבוצות משני צדי היעד?

**מה לחפש בטבלה:** שוויון 1/2 נבדק אצל האלופה והסגנית הרשמיות; בירידה אצל השורדת האחרונה והיורדת הראשונה לפי המיקום הרשמי. בקו העליון משווים את שני המקומות לפי ניקוד השלב הסדיר.

**התשובה הכדורגלנית:** שוויון ספורטיבי נמצא פעם אחת בקו העליון וב־5 עונות בקו הירידה; שתי קבוצות עם אותן נקודות עשויות לסיים משני צדי היעד. איזה חוק הכריע את המיקום הרשמי נבדוק ב־03 עם rules.py.

</div>

In [ ]:
ties = critical_ties(tables, upper_sizes=upper_sizes)
display(ties)

<div dir="rtl">

### כמה עונות הסתיימו בשוויון קריטי?
**השאלה:** עד כמה נחוצים שוברי שוויון בסימולציה?

**מה לחפש בטבלה:** n_applicable הוא מספר העונות שבהן היעד קיים; רק 15 עונות כוללות קו עליון.

**התשובה הכדורגלנית:** נצפו 5 שוויונות בקו הירידה, 1 בקו העליון ו־0 בין האלופה לסגנית בנקודות הספורטיביות. שוברי שוויון חשובים גם כשאין שוויון בפסגה.

</div>

In [ ]:
tie_counts = critical_ties_summary(ties)
display(tie_counts)

<div dir="rtl">

### כמה שווה הבית בכל עונה?
**השאלה:** האם יתרון הבית יציב לאורך השנים?

**מה לחפש בטבלה:** ההפרש בנקודות למשחק ובשערים משווה בית לחוץ; עמודות רווח הסמך מראות את אי־הוודאות בשיעור ניצחונות הבית.

**התשובה הכדורגלנית:** ממוצע פער הנקודות בית–חוץ ירד מכ־+0.38 ב־2009/10–2014/15 לכ־+0.14 ב־2015/16–2023/24; ממוצע שיעור ניצחונות הבית ירד מכ־43% לכ־39%. המשמעות היא יתרון בית נמוך יותר בתקופה המאוחרת, לא יתרון קבוע לכל השנים.

</div>

In [ ]:
home_seasons = home_advantage_by_season(tables)
display(
    home_seasons[
        [
            "n",
            "home_win_rate",
            "draw_rate",
            "away_win_rate",
            "home_win_rate_ci_lower",
            "home_win_rate_ci_upper",
            "goals_pm_gap",
            "ppm_gap",
        ]
    ]
    .tail(8)
    .round(3)
)

home_periods = home_advantage_periods(tables)
display(home_periods.round(3))

<div dir="rtl">

### גרף B3 — הביתיות לאורך הזמן
**השאלה:** באילו עונות ניצחונות בית היו שכיחים יותר?

**מה לחפש בגרף:** טווחים חופפים דורשים זהירות. סימון 2019/20 ו־2020/21 מציין את תקופת הקורונה ואת שאלת הקהל — הקשר היסטורי לאימות, ולא משתנה שנמדד בדאטה שלנו.

**התשובה הכדורגלנית:** פער הבית–חוץ ירד בממוצע עונות מכ־+0.38 לכ־+0.14 נקודות למשחק (43% מול 39% ניצחונות בית). ב־2020/21 הפער היה כ־+0.09: היא אינה חריגה שממנה התחילה הירידה, כי כבר ב־2015/16 הוא היה כ־+0.04. ההקשר של משחקים בלי קהל אינו מוכיח סיבתיות. ב־goals נבדוק ביתיות משתנה בזמן או משקל לעונות אחרונות, ללא בחירת שיטה כאן. [goals]

</div>

In [ ]:
chapter_b_chart("home", home_seasons, home_periods).show()

<div dir="rtl">

### האם העידנים נראים דומים?
**השאלה:** האם ליגה של 12, 16 או 14 קבוצות מציגה אותם שערים ואותו איזון?

**מה לחפש בטבלה:** n_matches הוא מספר המשחקים; הפיזור מודד כמה רחוקים קצבי הנקודות של הקבוצות בתוך עונה.

**התשובה הכדורגלנית:** ממוצע השערים הוא כ־2.33, 2.58 ו־2.46 בעידני 12, 16 ו־14 הקבוצות בהתאמה. השינוי יכול לשקף גם תקופה וגם פורמט.

</div>

In [ ]:
eras = era_comparison(tables)
display(
    eras[
        [
            "n_matches",
            "n_seasons",
            "goals_per_match",
            "draw_rate",
            "zero_zero_rate",
            "home_win_rate",
            "big_margin_rate",
            "std_ppm",
            "title_margin_ppm",
        ]
    ].round(3)
)

<div dir="rtl">

### גרף B4 — עידנים שונים, שאלת אימון פתוחה
**השאלה:** האם מודל הטבלה ילמד מכל העונות או רק מ־2012/13?

**מה לחפש בגרף:** השוו שערים, ניצחונות בהפרש של שלושה ומעלה ופיזור נקודות. שני המדדים הראשונים מציגים טווחי אי־ודאות; הפיזור הוא ממוצע תיאורי של עונות.

**התשובה הכדורגלנית:** בעידן 16 הקבוצות נצפו 2.58 שערים למשחק ו־15% הפרשים של 3 ומעלה, מול 2.46 ו־14% בעידן 14; יש שינוי בתמהיל המשחקים, אך הבחירה בין כל העונות לבין התחלה ב־2012/13 תיבדק במודל עצמו. [table_model]

</div>

In [ ]:
chapter_b_chart("eras", eras).show()

<div dir="rtl">

### סיכום העונה הקיים — גם פנדלים ואדומים
**השאלה:** אילו תופעות משתנות לאורך העונות?

**מה לחפש בטבלה:** אלה הערכים שכבר חושבו ב־season_summary, ללא חישוב מחדש. שערי פנדל אינם כוללים החמצות; אדומים אינם כוללים את כל הצהובים.

**התשובה הכדורגלנית:** שערי פנדל למשחק עלו מכ־0.157 ב־2006/07 לכ־0.247 ב־2023/24, בעוד האדומים ירדו מכ־0.384 לכ־0.283. זה מתאר אירועים שנרשמו במקור, ולא כל בעיטת פנדל או כרטיס שנשרק.

</div>

In [ ]:
season_trends = tables.season_summary.loc[list(DEVELOPMENT_SEASONS)]
display(
    season_trends[
        [
            "n_matches",
            "goals_per_match",
            "draw_rate",
            "zero_zero_rate",
            "penalties_per_match",
            "red_cards_per_match",
        ]
    ]
    .tail(8)
    .round(3)
)

<div dir="rtl">

### גרף B5 — שערים ותוצאות סגורות לאורך השנים
**השאלה:** האם תיקו ו־0:0 נשארו באותה שכיחות?

**מה לחפש בגרף:** השוו מגמות בין העונות; אין להסיק מעונה אחת שהליגה השתנתה לתמיד. מספר המשחקים מופיע בריחוף.

**התשובה הכדורגלנית:** בין 2006/07 ל־2023/24 שיעור התיקו ירד מ־30.8% ל־23.8%, ו־0:0 מ־12.1% ל־6.25%; תוצאות סגורות פחות שכיחות בשתי נקודות הקצה האלה, אך הגרף מראה שהשינוי אינו ירידה רצופה בכל עונה. [goals, scores]

</div>

In [ ]:
chapter_b_chart("season_trends", season_trends).show()

<div dir="rtl">

### איזון תחרותי
**השאלה:** עד כמה גדול הפער בין קבוצות באותה עונה?

**מה לחפש בטבלה:** std_ppm מתאר פיזור, top_bottom_ppm_gap את הפער בין קצבי הנקודות הגבוה והנמוך; פער האליפות מתייחס לאלופה ולסגנית הרשמיות.

**התשובה הכדורגלנית:** פיזור הנקודות למשחק בין הקבוצות נע בין כ־0.30 לכ־0.49; שתי עונות עם מאבק אליפות צמוד יכולות להציג פערים שונים בשאר הליגה.

</div>

In [ ]:
balance = competitive_balance(tables)
display(balance.tail(8).round(3))

<div dir="rtl">

### מי זכתה באליפויות?
**השאלה:** עד כמה התארים מרוכזים במעט מועדונים?

**מה לחפש בטבלה:** כל האליפויות מבוססות על המיקום הרשמי, וכל 18 עונות הפיתוח מיוצגות.

**התשובה הכדורגלנית:** שישה מועדונים זכו; מכבי ת״א ומכבי חיפה זכו יחד ב־11 מ־18 התארים.

</div>

In [ ]:
champions = champions_summary(tables)
display(champions)

<div dir="rtl">

### גרף B6 — פיזור ותארים
**השאלה:** איך נראים יחד התנודתיות בין עונות וריכוז האליפויות?

**מה לחפש בגרף:** משמאל פיזור נקודות בין הקבוצות; מימין מספר התארים לכל מועדון.

**התשובה הכדורגלנית:** מכבי ת״א ומכבי חיפה זכו יחד ב־11 מתוך 18 האליפויות; פיזור הנקודות בין הקבוצות נע בין 0.30 ל־0.49. ריכוז התארים אינו אומר שכל עונה הייתה מרוץ חד־צדדי. [table_model, season_sim]

</div>

In [ ]:
chapter_b_chart("balance", balance, champions).show()

<div dir="rtl">

### יום ושעה — מי משחקת מתי?
**השאלה:** האם שיעורי הבית והשערים נראים שונים לפי יום או שעת פתיחה?

**מה לחפש בטבלה:** שעות הן בשעון ישראל. שימו לב למספר המשחקים ולטווחי אי־הוודאות; קבוצות ושידורים עשויים להיות משובצים באופן לא אקראי.

**התשובה הכדורגלנית:** אחרי 20:00 נצפו כ־2.55 שערים למשחק מול כ־2.41 בין 18:00 ל־19:59, וטווחי אי־הוודאות חופפים. קבוצות ושיבוצים שונים יכולים להסביר את ההבדל; זו אינה הוכחה ששעה גורמת לשערים.

</div>

In [ ]:
calendar = calendar_effects(tables)
display(
    calendar.loc[
        ["weekday", "kickoff_hour"],
        [
            "n",
            "goals_per_match",
            "goals_per_match_ci_lower",
            "goals_per_match_ci_upper",
            "home_win_rate",
            "home_win_rate_ci_lower",
            "home_win_rate_ci_upper",
            "draw_rate",
        ],
    ].round(3)
)

<div dir="rtl">

### גם החודש משנה את קבוצת המשחקים
**השאלה:** האם חודשי סוף העונה נראים שונים מחודשי החורף?

**מה לחפש בטבלה:** יוני ויולי כוללים מעט משחקים; ביולי n=12, ולכן אין להסיק ממנו מסקנה. סוף העונה כולל גם פלייאוף והרכב יריבות שונה.

**התשובה הכדורגלנית:** בינואר נצפו כ־2.39 שערים למשחק (555 משחקים), ובמאי כ־2.72 (380 משחקים). תמהיל הפלייאוף והיריבות שונה, ולכן ההבדל אינו הוכחה להשפעה של החודש עצמו.

</div>

In [ ]:
display(
    calendar.xs("month", level="dimension")
    .reindex([str(m) for m in range(1, 13)])[
        [
            "n",
            "goals_per_match",
            "home_win_rate",
            "home_win_rate_ci_lower",
            "home_win_rate_ci_upper",
            "draw_rate",
        ]
    ]
    .round(3)
)

<div dir="rtl">

### גרף B7 — ימים ושעות עם אי־ודאות
**השאלה:** אילו הבדלים נשארים משכנעים כשמציגים גם את הטווחים?

**מה לחפש בגרף:** חפיפת טווחים וכמות המשחקים בריחוף. הרבה השוואות עלולות לייצר הבדל מקרי; אין דירוג של יום ״טוב״ או ״רע״.

**התשובה הכדורגלנית:** אחרי 20:00 נצפו כ־2.55 שערים מול כ־2.41 בשעות 18:00–19:59, בתוך טווחי אי־הוודאות החופפים; ההפרש כ־0.14 אינו בסיס לקביעת פיצ׳ר או כלל חיזוי. [הבנת הליגה]

</div>

In [ ]:
chapter_b_chart("calendar", calendar).show()

<div dir="rtl">

### משחקים נדחים מול שאר המשחקים
**השאלה:** האם המשחקים שנדחו אחרי תחילת המחזור הבא באותו שלב נראים שונים?

**מה לחפש בטבלה:** רק 49 משחקים נדחים מול 4,266 אחרים; השיעורים של הקבוצה הקטנה פחות יציבים. הפלייאופים המקבילים כבר אינם מסומנים בטעות.

**התשובה הכדורגלנית:** הבית ניצח בכ־53.1% מהנדחים מול כ־40.3% מהאחרים, אך המדגם הקטן ותמהיל היריבות אינם מאפשרים טענה שדחייה גורמת ליתרון.

</div>

In [ ]:
postponed = postponed_summary(tables)
display(
    postponed[
        [
            "n",
            "goals_per_match",
            "home_win_rate",
            "home_win_rate_ci_lower",
            "home_win_rate_ci_upper",
            "draw_rate",
            "away_win_rate",
        ]
    ].round(3)
)

<div dir="rtl">

## מה למדנו — פעימה B

1. בעידן 14 הקבוצות השורדת בעלת הניקוד הנמוך צברה חציון של **34 נקודות** (טווח **28–37**, אחרי 32–33 משחקים שנכללו), והאלופה חציון **83.5**; אלה ספים היסטוריים ולא הבטחת הישרדות או אליפות. קו העליון הוא יעד מוקדם יותר, אחרי השלב הסדיר. [motivation, table_model]
2. נצפו **5 עונות** עם שוויון ספורטיבי בקו הירידה ו־**1 מתוך 15** בקו העליון; נצטרך חוקי שוברי שוויון מתועדים ב־03, בלי להסיק כאן מה הכריע את המיקום הרשמי. [season_sim, table_model]
3. הביתיות ירדה: בממוצע עונות פער הבית–חוץ היה **+0.38** נקודות למשחק ב־2009/10–2014/15 מול **+0.14** ב־2015/16–2023/24; ניצחונות הבית כ־**43% מול 39%**. ב־2020/21 הפער היה **+0.09**, כך שעונת הקורונה אינה חריגה שממנה התחילה הירידה. לא נניח ביתיות קבועה: נבדוק ביתיות משתנה בזמן או משקל לעונות אחרונות, ללא החלטת מודל או טענה סיבתית על הקהל כאן. [goals]
4. בעידני 12, 16 ו־14 הקבוצות נצפו **2.33, 2.58 ו־2.46 שערים למשחק** בהתאמה; נצטרך להשוות אימון על כל העונות מול התחלה ב־2012/13, בלי לבחור חלון על סמך התיאור הזה בלבד. [table_model, goals]
5. **6 מועדונים** חילקו ביניהם 18 אליפויות, ומכבי ת״א ומכבי חיפה זכו ב־**11**; ריכוז התארים אינו מחליף את התמונה של פיזור הנקודות בין הקבוצות בכל עונה. [table_model, season_sim]
6. המשחקים אחרי 20:00 הציגו כ־**2.55 שערים** לעומת **2.41** בין 18:00 ל־19:59, אך קבוצות ושיבוצים שונים עשויים להסביר זאת; גם **49** המשחקים הנדחים הם מדגם קטן. אלה קשרים לבדיקה, ללא החלטת פיצ׳רים כאן. [הבנת הליגה]

### מה מגיע עכשיו?
פעימה B נעצרת לסקירת Claude Code ועמית. פעימה C תתחיל רק לאחר אישור ומיזוג בלולאת העבודה הרגילה.

</div>

<div dir="rtl">

## C — מהלך העונה והטבלה

> ### איך לקרוא את הפרק
> **אחרי k משחקים אינו תאריך:** כל קבוצה נמדדת אחרי אותו מספר משחקים שלה, לפי תאריך ואז game_id. אם משחק נדחה, המשחק ה־k של קבוצה אחת עשוי להיות אחרי המשחק ה־k של אחרת. התצוגה תיאורית בדיעבד; טבלה אמיתית לפי as_of תיבנה ב־03. אין להשתמש בדירוג הזה כקלט טרום־משחק.
>
> **הדירוג:** נקודות → הפרש שערים → שערי זכות; שוויון מלא מקבל דירוג ממוצע. זה כלל תיאורי, לא מימוש כל שוברי השוויון הרשמיים. games_short מסמן קבוצה שאין לה k משחקים; סוחבים את הניקוד האחרון שלה בלי להמציא משחק או נקודה. בגרף מרוץ הנקודות לא מציירים את ההמשך הזה כמשחקים נוספים.
>
> **שוויונות בשיעורים — החלטת עמית:** אם m קבוצות מובילות בשוויון מלא, קרדיט האליפות הוא מספר האלופות ביניהן חלקי m; בתחתית מספר היורדות חלקי m. כל העונות נשארות במכנה והממוצע נותן משקל שווה לעונה. על קו העליון, כל קבוצה בשוויון מקבלת משקל של מספר המקומות הפנויים חלקי גודל השוויון; ההצלחה המשוקללת חלקי גודל העליון ממוצעת בעונות עם פיצול. rank ממוצע נשמר. n_tied_leader/n_tied_bottom מציינים עונות עם שוויון; גרסת excl_ties מוצגת לרגישות בלבד. מתאם לא מוגדר בטבלה שכולה שווה נשאר חסר.
>
> **נקודות ומיקום:** 3/1/0 מכל המשחקים שנכללו, ללא עונשים וללא חצייה; המיקום הסופי והשלב הם תוויות רשמיות. בעונות 11–12 הייתה חציית נקודות, וב־2009/10 היא הכריעה את האליפות. המעבר בין דירוג ספורטיבי בפיצול לבין מיקום רשמי אינו כולו שינוי כוח.
>
> **אי־ודאות:** Wilson של 95% לספירות בינריות. כשיש קרדיטים חלקיים, רווח הממוצע מגיע מ-bootstrap של עונות; כשהקרדיטים כולם 0/1 נשמר Wilson. גם ממוצעי מתאם ושערים משתמשים בדגימת עונות, 2,000 דגימות וזרע קבוע. ci_method מציין איזו שיטה שימשה. גרסת הרגישות ללא שוויונות בינרית ולכן משתמשת ב-Wilson. אין תיקון להשוואות מרובות או הוכחת סיבתיות. בעידני 12 ו־16 קבוצות רק 3 עונות, ובפלייאוף האמצעי רק 12 משחקים; אלה תיאורים עם מגבלת מדגם, לא בסיס להכרעה חזקה.
>
> **מחזורים אחרונים והפתעות:** שלושה מספרי המחזור האחרונים בכל עונה ובאותו stage, לא שלושת התאריכים האחרונים. הפתעה היא ניצחון של הקבוצה עם ppm נמוך יותר בעונה עד תאריך קודם ממש. מצב זה נשלף מסיכומי המשחק ה־k של כל קבוצה עצמה; אין שימוש בדירוג שאר הקבוצות בעתיד. בלי היסטוריה לשני הצדדים או עם ppm שווה לא נכנסים למכנה ההפתעות. תיקו בין צדדים בעלי ppm שונה נשאר אי־הפתעה.
>
> **היקף:** 4,315 משחקי פיתוח בלבד. אין features, אימון או שינוי ב-clean/raw. המוטיבציה המתמטית המלאה תיבדק ב־04 (VISION §7, שאלה 6א). אין בפעימה C ניתוחי מחצית או תזמון שערים.

**השאלה:** באיזה קנה מידה נכיר את מהלך העונה?

**מה לחפש בטבלה:** אותו בסיס ספורטיבי ושלבים מובחנים, עם מגבלות התקופה.

**התשובה הכדורגלנית:** אחרי 10 משחקים המובילה זכתה ב־12 מתוך 18 עונות; בפיצול זכתה המובילה ב־14 מתוך 15. הטבלה מתייצבת בהמשך העונה, אבל חוקי התקופה והיריבות בפלייאוף עדיין משנים את התמונה.

</div>

In [ ]:
from ligat.analysis.league_stats import (
    final_rounds_check,
    playoff_form,
    playoff_form_summary,
    season_points_race,
    settling_sensitivity,
    split_movement,
    split_summary,
    stage_comparison,
    standings_by_match_number,
    table_settling,
)
from ligat.viz.season_course import chapter_c_chart

display(
    pd.DataFrame(
        {
            "מבט": ["התייצבות", "תזוזה", "מוטיבציה פשוטה"],
            "בסיס": ["משחקים לקבוצה", "סוף סדיר מול סיום רשמי", "3 מחזורים אחרונים בכל שלב"],
        }
    )
)

<div dir="rtl">

### הטבלה אחרי אותו מספר משחקים
**השאלה:** איך נראה המיקום כשכל קבוצה השלימה 10 משחקים?

**מה לחפש בטבלה:** זו דוגמת 2023/24; הנקודות, ההפרש ושערי הזכות קובעים את הדירוג התיאורי. שוויון מלא אינו נשבר בעזרת מזהה קבוצה.

**התשובה הכדורגלנית:** ב־2023/24, אחרי 10 משחקים לכל קבוצה, מכבי ת״א מובילה עם 28 נקודות מול 23 של מכבי חיפה — פער של 5 נקודות. אלה נקודות על המגרש, ללא עונשים; המשחק העשירי של כל קבוצה לא בהכרח נערך באותו יום.

</div>

In [ ]:
standings = standings_by_match_number(tables)
display(
    standings.xs((EXAMPLE_SEASON, 10), level=["season_id", "k"])
    .join(tables.team_seasons.xs(EXAMPLE_SEASON, level="season_id")[["team_name"]])
    .sort_values("rank")
    .head(8)
    .round(3)
)

<div dir="rtl">

### כמה הסדר המוקדם דומה לסוף?
**השאלה:** מתי הטבלה מתחילה להיראות כמו טבלת הסיום?

**מה לחפש בטבלה:** מתאם קרוב ל־1 מתאר סדר דומה. n_seasons הוא מספר העונות, והמכנים של שיעורי המובילה והאחרונה כוללים את כל העונות, גם כשהקרדיט חלקי.

**התשובה הכדורגלנית:** אחרי 10 משחקים המתאם הממוצע הוא כ־0.60 בעידן 12 הקבוצות, 0.85 ב־16 ו־0.68 ב־14; יש כבר קשר לסיום, אך לא ידיעה של התוצאה הסופית.

</div>

In [ ]:
settling = table_settling(tables, upper_sizes=upper_sizes)
display(
    settling.xs(10, level="k")[
        [
            "n_seasons",
            "spearman_mean",
            "spearman_ci_lower",
            "spearman_ci_upper",
            "leader_won_title_rate",
            "leader_won_title_rate_n",
            "bottom_relegated_rate",
            "bottom_relegated_rate_n",
            "in_upper_line_stayed_rate",
        ]
    ].round(3)
)

<div dir="rtl">

### האם השוויונות משנים את התמונה?
**השאלה:** האם הכללת קרדיט חלקי נותנת מסקנה שונה מהוצאה של עונות בשוויון?

**מה לחפש בטבלה:** זו השוואה בעידן 14 הקבוצות בנקודות נבחרות; excl_ties היא רגישות בלבד, ו-NA שם מציין שלא נשארו עונות בלי שוויון. המדד הראשי תמיד כולל את כולן.

**התשובה הכדורגלנית:** אחרי 10 משחקים שני שיעורי האליפות ושני שיעורי הירידה זהים — לא היו שוויונות בראש או בתחתית באותה נקודה. הפער המרבי בין הגרסאות בכל עידן ו-k הוא כ־2.3 נקודות אחוז לשני המדדים, כך שהממצאים קרובים בדאטה הזה; שתי הגרסאות ומספר השוויונות נשארים גלויים. [table_model, season_sim]

</div>

In [ ]:
display(
    settling.xs("14_teams", level="era")
    .loc[
        [1, 5, 10, 20],
        [
            "n_seasons",
            "leader_won_title_rate",
            "leader_won_title_excl_ties",
            "bottom_relegated_rate",
            "bottom_relegated_excl_ties",
            "n_tied_leader",
            "n_tied_bottom",
        ],
    ]
    .round(3)
)

sensitivity = settling_sensitivity(tables, upper_sizes=upper_sizes)
display(sensitivity.round(4))

<div dir="rtl">

### גרף C1 — התייצבות הטבלה
**השאלה:** האם ההיכרות עם הסדר הסופי עולה ככל שמשחקים יותר?

**מה לחפש בגרף:** השוו את העידנים ואת טווחי אי־הוודאות. כל נקודה היא ממוצע עונות, ולא טבלה מתאריך מסוים.

**התשובה הכדורגלנית:** בעידן 14 הקבוצות המתאם אחרי 10 משחקים הוא כ־0.68; מיקום מוקדם נותן מידע אך משאיר מקום לתזוזה, ולכן אינו תחליף לסימולציה. [table_model, season_sim]

</div>

In [ ]:
chapter_c_chart("settling", settling).show()

<div dir="rtl">

### גרף C2 — ההובלה והאחרונה אינן גורל
**השאלה:** האם המובילה והאחרונה אחרי k משחקים מסיימות באותו יעד?

**מה לחפש בגרף:** קו כחול לאליפות וקו כתום לירידה, בנפרד לכל עידן. בריחוף מופיע מספר העונות כולל שוויונות; בדגימות של 3 עונות הרווח רחב.

**התשובה הכדורגלנית:** אחרי 10 משחקים המובילה זכתה ב־12 מתוך 18 עונות, והאחרונה ירדה ב־6 מתוך 18; ההובלה המוקדמת אינה תואר, והמקום האחרון אינו ירידה ודאית. [table_model, season_sim]

</div>

In [ ]:
chapter_c_chart("leaders", settling).show()

<div dir="rtl">

### סוף השלב הסדיר מול הסיום
**השאלה:** מי עלתה ומי ירדה במיקום אחרי הפיצול?

**מה לחפש בטבלה:** move חיובי הוא שיפור; שלילי הוא ירידה. המיקום בפיצול ספורטיבי והסיום רשמי, ולכן גם חוקים ועונשים יכולים להסביר חלק מהפער.

**התשובה הכדורגלנית:** בעליון נצפו תזוזות של עד 5 מקומות לשני הכיוונים, ובתחתון עד 4; גם כשההובלה יציבה, סדר הקבוצות בתוך הפלייאוף עדיין עשוי להשתנות.

</div>

In [ ]:
movement = split_movement(tables, upper_sizes=upper_sizes)
display(
    movement.xs(EXAMPLE_SEASON, level="season_id")[
        [
            "team_name",
            "rank_at_split",
            "final_position",
            "move",
            "group",
            "games_short",
            "points_halved",
        ]
    ]
    .head(8)
    .round(3)
)

<div dir="rtl">

### סיכום התזוזה — כמה נשארו במקום?
**השאלה:** באיזה פלייאוף הסדר זז יותר?

**מה לחפש בטבלה:** n הוא קבוצות־עונות, בעוד leader_n הוא כל העונות עם פיצול ו-leader_credit_sum הוא סכום קרדיטים. האמצעי כולל רק 8 קבוצות־עונות בשתי עונות.

**התשובה הכדורגלנית:** 56 מ־92 קבוצות־העונות בעליון נשארו באותו מיקום, לעומת 38 מ־116 בתחתון; התחתון זז יותר לפי המדד הזה, בלי להסיק מה גרם לתזוזה.

</div>

In [ ]:
movement_summary = split_summary(tables, upper_sizes=upper_sizes)
display(
    movement_summary[
        [
            "n",
            "improved",
            "unchanged",
            "declined",
            "move_min",
            "move_max",
            "leader_credit_sum",
            "leader_n",
            "leader_at_split_won_title_rate",
        ]
    ].round(3)
)

<div dir="rtl">

### גרף C3 — התנועה אחרי הפיצול
**השאלה:** האם מי שהובילה בפיצול זכתה, וכמה זזו הקבוצות האחרות?

**מה לחפש בגרף:** כל נקודה היא קבוצה־עונה; תזוזה חיובית מציינת שיפור. שנת 2009/10 כוללת את החצייה ההיסטורית.

**התשובה הכדורגלנית:** המובילה בפיצול זכתה ב־14 מתוך 15 עונות; החריג הוא 2009/10, שבה החצייה שינתה את בסיס המרוץ. את חוקי הפיצול צריך לשחזר בסימולציה. [season_sim]

</div>

In [ ]:
chapter_c_chart("movement", movement, movement_summary).show()

<div dir="rtl">

### האם משחקי הפלייאוף שונים?
**השאלה:** האם השערים והתוצאות הסגורות נראים אחרת בכל שלב?

**מה לחפש בטבלה:** n_matches שונה מאוד: 3,492 בסדיר, 417 בעליון, 394 בתחתון ורק 12 באמצעי. אלה יריבות ותקופות שונות.

**התשובה הכדורגלנית:** בעליון נצפו 2.70 שערים למשחק מול 2.43 בסדיר, ותיקו בכ־24% מול כ־28%; זה קשר לשלב ולתמהיל היריבות, לא הוכחה למוטיבציה גבוהה יותר.

</div>

In [ ]:
stages = stage_comparison(tables)
display(
    stages[
        [
            "n_matches",
            "goals_per_match",
            "draw_rate",
            "zero_zero_rate",
            "one_one_rate",
            "btts_rate",
            "home_win_rate",
            "big_margin_rate",
        ]
    ].round(3)
)

<div dir="rtl">

### גרף C4 — קצב השערים והפלייאוף
**השאלה:** עד כמה ההבדלים בין השלבים בולטים לצד טווחי אי־הוודאות?

**מה לחפש בגרף:** שערים, תיקו, 0:0 ושתי הקבוצות מבקיעות; n נמצא בריחוף. אין מסקנה חזקה מהאמצעי עם 12 משחקים.

**התשובה הכדורגלנית:** שתי הקבוצות הבקיעו בכ־52% ממשחקי העליון מול כ־46% בסדיר; מודל התוצאות והסימולציה יצטרכו לבדוק אם ההקשר מוסיף מעבר לחוזק היריבות. [scores, season_sim]

</div>

In [ ]:
chapter_c_chart("stages", stages).show()

<div dir="rtl">

### מי העלתה קצב בפלייאוף?
**השאלה:** האם אותה קבוצה צוברת נקודות בקצב אחר אחרי הפיצול?

**מה לחפש בטבלה:** ppm_change חיובי מציין שיפור; זו דוגמה של 2023/24, מסודרת לשם קריאה. מספר משחקי הפלייאוף קטן ממספר משחקי הסדיר.

**התשובה הכדורגלנית:** בעליון הקצב יורד בממוצע בכ־0.41 נקודות למשחק; בתחתון הוא עולה בכ־0.35. בפלייאוף כל היריבות בעליון חזקות ובתחתון חלשות יותר, ולכן חשוב להשוות בתוך קבוצת הפלייאוף ולא לספור את כל השיפורים יחד. קשר, לא סיבה.

</div>

In [ ]:
form = playoff_form(tables)
display(
    form.xs(EXAMPLE_SEASON, level="season_id")
    .sort_values("ppm_change")[
        [
            "team_name",
            "final_stage",
            "n_regular",
            "n_playoff",
            "ppm_regular",
            "ppm_playoff",
            "ppm_change",
        ]
    ]
    .head(8)
    .round(3)
)

<div dir="rtl">

### שינוי הקצב לפי קבוצת פלייאוף
**השאלה:** האם השינוי שונה בעליון ובתחתון?

**מה לחפש בטבלה:** n הוא מספר קבוצות־העונות; n_seasons מספר העונות. הממוצע והחציון מתארים את שינוי הנקודות למשחק. רווחי הסמך דוגמים עונות שלמות, כפי שהוסבר בתחילת הפרק.

**התשובה הכדורגלנית:** העליון יורד בכ־0.41 נקודות למשחק (92 קבוצות־עונות), והתחתון עולה בכ־0.35 (116). באמצעי כ־0.145, אבל רק 8 קבוצות־עונות בשתי עונות — מדגם קטן במיוחד. שינוי היריבות הוא הסבר אפשרי, לא מסקנה סיבתית.

</div>

In [ ]:
form_summary = playoff_form_summary(tables)
display(form_summary.round(3))

<div dir="rtl">

### גרף C5 — קצב לפני ואחרי
**השאלה:** מי נמצאת מעל קו הקצב הזהה?

**מה לחפש בגרף:** מעל האלכסון קצב גבוה יותר בפלייאוף; מתחתיו נמוך יותר. בריחוף נמצאים שם, עונה ומספר המשחקים.

**התשובה הכדורגלנית:** העליון מאבד בממוצע כ־0.41 נקודות למשחק והתחתון מוסיף כ־0.35. שינוי חוזק היריבות יכול להסביר שינוי קצב בלי שינוי בכוח הקבוצה. ל־table_model אסור להמשיך את קצב הסדיר כפי שהוא; ב־season_sim יש להביא בחשבון את חוזק היריבות בפלייאוף. קשר, לא סיבה. [table_model, season_sim]

</div>

In [ ]:
chapter_c_chart("form", form, form_summary).show()

<div dir="rtl">

### שלושת המחזורים האחרונים
**השאלה:** האם לקראת סוף כל שלב רואים יותר שערים או הפתעות?

**מה לחפש בטבלה:** n הוא כל המשחקים, upset_n רק משחקים עם היסטוריה לשני הצדדים ו־ppm שונה. באמצעי כל 12 המשחקים נמצאים בשלושת המחזורים האחרונים, ולכן אין קבוצת השוואה ״שאר״.

**התשובה הכדורגלנית:** בסוף נצפו פחות תיקו בכל שלב שיש בו השוואה: סדיר כ־22% מול 28%, עליון 16% מול 28%, ותחתון 22% מול 29%. הפתעות בפלייאוף היו כ־30% מול 23% בעליון וכ־42% מול 33% בתחתון, בקרב הזכאיות. זה רמז למחקר, לא הוכחת מוטיבציה.

</div>

In [ ]:
last_rounds = final_rounds_check(tables)
display(
    last_rounds[
        [
            "n",
            "goals_per_match",
            "draw_rate",
            "home_win_rate",
            "upset_n",
            "upset_count",
            "upset_rate",
        ]
    ].round(3)
)

<div dir="rtl">

### גרף C6 — רמזים למחקר מוטיבציה
**השאלה:** האם יש שינוי בתוצאות בשלבים המאוחרים?

**מה לחפש בגרף:** השוו את התקופות בתוך אותו שלב, כולל הרווחים והמכנים. הפתעה משתמשת רק בניקוד שהיה ידוע לפני המשחק, אך אינה מזהה מי חסרת עניין מתמטית.

**התשובה הכדורגלנית:** פחות תיקו בסוף, בכל השלבים עם קבוצת השוואה: סדיר 22% מול 28%, עליון 16% מול 28%, תחתון 22% מול 29%. הפתעות בעליון 30% מול 23% ובתחתון 42% מול 33%. הרמז מזמין מחקר מוטיבציה ב־04 לפי טבלה as_of, ולא מוכיח סיבה. [motivation; VISION §7, שאלה 6א]

</div>

In [ ]:
chapter_c_chart("last_rounds", last_rounds).show()

<div dir="rtl">

### דוגמה: מרוץ 2023/24
**השאלה:** איך נצברו הנקודות לאורך העונה שבחרנו?

**מה לחפש בטבלה:** תמונת הקבוצות אחרי 26 המשחקים הראשונים שלהן, סמוך לפיצול. זה אינו snapshot מתאריך משותף.

**התשובה הכדורגלנית:** בעליון סיימו לאחר 36 משחקים ובתחתון לאחר 33; השוואת קווים לפי k מפרידה קצב ממשחקים חסרים, אבל אינה מראה מי ידעה את המיקום בזמן אמת.

</div>

In [ ]:
race = season_points_race(tables, [EXAMPLE_SEASON])
display(
    race.xs((EXAMPLE_SEASON, 26), level=["season_id", "k"])
    .sort_values("rank")[["team_name", "points", "goal_difference", "games_played", "rank"]]
    .head(8)
    .round(3)
)

<div dir="rtl">

### גרף C7 — מרוץ נקודות ספורטיבי
**השאלה:** מתי התרחב הפער בין הקבוצות בדוגמת 2023/24?

**מה לחפש בגרף:** קו הפיצול ב־26, ומספר משחקים שונה לכל פלייאוף. שורות games_short אינן מצוירות כמשחקים נוספים; אין הורדות נקודות.

**התשובה הכדורגלנית:** מכבי ת״א סיימה עם 85 נקודות על המגרש ומכבי חיפה עם 76; הקווים מראים את צבירת הפער, לא את הסתברותו בתחילת העונה. [table_model, season_sim]

</div>

In [ ]:
chapter_c_chart("race", race).show()

<div dir="rtl">

## מה למדנו — פעימה C

1. אחרי **10 משחקים**, המתאם לסיום הוא כ־**0.60–0.85** בין העידנים, וכ־**0.68** בעידן 14 הקבוצות; הטבלה כבר נותנת מידע אבל נשארת אי־ודאות. זו השוואה לפי מספר משחקים, לא טבלה as_of ולא פיצ׳ר שכבר נבנה. [table_model, season_sim]
2. אחרי 10 משחקים המובילה זכתה ב־**12 מתוך 18** עונות והאחרונה ירדה ב־**6 מתוך 18**; בפיצול המובילה זכתה ב־**14 מתוך 15**. החריג 2009/10 מזכיר שחציית הנקודות היא חלק מהחוקים שצריך לשחזר. [season_sim]
3. **56 מ־92** קבוצות־העונות בעליון נשארו באותו מיקום מול **38 מ־116** בתחתון; בפלייאוף זזים גם כשזהות האלופה כבר קשורה מאוד למובילה בפיצול. קצב העליון יורד בממוצע בכ־**0.41** נקודות למשחק וקצב התחתון עולה בכ־**0.35**; באמצעי כ־**0.145** בלבד, על 8 קבוצות־עונות. בעליון כל היריבות חזקות ובתחתון חלשות יותר: קשר, לא סיבה. ל־table_model אסור להמשיך את קצב הסדיר כפי שהוא; season_sim צריך להביא בחשבון את חוזק היריבות בפלייאוף. [table_model, season_sim]
4. בפלייאוף העליון נצפו **2.70 שערים** למשחק מול **2.43** בסדיר, ושתי הקבוצות הבקיעו בכ־**52% מול 46%**; זו בדיקת הקשר למודלי התוצאות והסימולציה, לא טענה על סיבה. [scores, season_sim]
5. **פחות תיקו בסוף, בכל השלבים עם קבוצת השוואה:** סדיר **22% מול 28%**, עליון **16% מול 28%**, תחתון **22% מול 29%**. הפתעות בפלייאוף: עליון **30% מול 23%**, תחתון **42% מול 33%**, בקרב משחקים עם עבר ידוע ו־ppm שונה. אלה רמזים למחקר מוטיבציה ב־04, לא הוכחה. הגרסה הפשוטה אינה מזהה ״חסרת עניין מתמטית״ — את הניתוח המלא נעשה ב־04 לפי טבלה as_of. [motivation]

### מה מגיע עכשיו?
פעימה C נעצרת לסקירת Claude Code ועמית. פעימה D תתחיל רק אחרי QA, מיזוג ובדיקת יציבות בלולאת העבודה.

</div>

<div dir="rtl">

## D — המשחק והשערים

> ### איך לקרוא את הפרק
> **שעון תיאורי 0–90:** תוספת המחצית הראשונה נשמרת בדקת המקור אבל משויכת לשעון 45; תוספת המחצית השנייה לשעון 90. שערים באותה דקה מסודרים לפי half, minute, goal_number, ומשך מצב של אפס דקות מותר. זה סדר מוסכם לתיאור, לא ידיעה של סדר האירועים בתוך אותה דקה.
>
> **החרגות:** מתוך 4,315 משחקים, 13 עם ht_conflict ועוד 864982 עם מבקיעים לא אמינים מוחרגים ממחצית, תזמון וציר זמן. נשארים 4,301 משחקים ו־8,602 שורות קבוצה×משחק. בכל פונקציה כזו נשמרים מספרי ההחרגות וסיבותיהן ב־attrs; הטבלה הבאה מציגה אותם. ספירות פנדל/עצמי אינן דורשות שיוך מחצית ולכן רק משחק המבקיעים הלא־אמינים מוחרג מהן.
>
> **מגבלות המקור:** יש שערי פנדל שהובקעו, לא פנדלים שנשרקו או הוחמצו. אין צהובים, בעיטות או xG; יש אדומים וצהוב שני. 12 אדומים ללא דקה נשמרים בספירות, אך מוחרגים מפירוק דקת האדום הראשון.
>
> **Poisson:** נקודת השוואה תיאורית עם ממוצעי הליגה, לא מודל שנאמן ולא תחזית. היא מתעלמת מחוזק קבוצות, ולכן הבדלי חוזק מסבירים חלק מהפערים. השוואה פורמלית במחברת 06. חלק גדול מהמתאם השלילי הכולל צפוי מהבדלי חוזק: חזקה מבקיעה יותר ויריבתה פחות. את התלות בתוך המשחק נבדוק רק אחרי ש-goals יקבע λ לכל צד, במחברת 06. לכן אין לפרש את המתאם הכולל כהוכחה לתלות שיורית. הרכיב המשותף ב-Poisson הדו־משתני הקלאסי מוסיף תלות חיובית; התאמתו תיבדק בהקשר המותנה, בלי לבחור כאן מודל.
>
> **אי־ודאות והשוואות מרובות:** לצד השיעורים n ורווחי Wilson של 95%; לממוצעים ולפיזור רווחי bootstrap של עונות שלמות (2,000 דגימות, זרע קבוע). n קטן מ־30 אינו בסיס למסקנה. הרווחים שוליים, לא תיקון לריבוי השוואות או לתלות של כל אירועי השערים. הדקות והנקודות לקבוצה×עונה הן תיאור מדויק של העונה, לא תחזית עם רווח סמך. רווח חסר כשאין לפחות שתי עונות.
>
> **קשר, לא סיבה:** במיוחד באדומים — הרחקה יכולה להיות גם תוצאה של מצב המשחק. אין כאן התאמה לחוזק, לביתיות או למצב לפני ההרחקה. גם זמן ביתרון ותוצאה סופית נמדדים באותו משחק.
>
> **נקודות מיתרון ומפיגור — הבהרת עמית:** איבוד מיתרון הוא 3 פחות הנקודות בפועל (0/2/3), רק אם הייתה הובלה. השגה מפיגור היא הנקודות בפועל (0/1/3), רק אם היה פיגור. אחרת NA. קבוצה שהובילה ואז פיגרה נספרת בשניהם. שוויון בתוך 10 דקות נמדד בשעון המקוצר, אחרי ספיגה שהשאירה בפיגור; NA אם לא הייתה ספיגה כזו.
>
> **פיתוח בלבד:** עונות 8–25, בלי features או אימון ובלי כתיבה ל-clean/raw.

**השאלה:** מה נוכל לתאר בביטחון בתוך המשחק?

**מה לחפש בגרף או בטבלה:** מספר המשחקים משתנה בין מפת התוצאות לניתוח ציר הזמן; כל החרגה מוצגת במפורש.

**התשובה הכדורגלנית:** 4,301 משחקים מאפשרים שחזור תקין של 90 דקות; 14 חריגי מקור אינם הופכים לשערים או למחציות מומצאים. [הבנת המשחק]

</div>

In [ ]:
from ligat.analysis.league_stats import (
    draw_types,
    first_goal_effect,
    first_goal_minute_distribution,
    game_timelines,
    goal_count_dispersion,
    goal_timing,
    halftime_transitions,
    observed_vs_poisson,
    penalty_owngoal_by_season,
    penalty_owngoal_stats,
    poisson_reference,
    red_card_stats,
    scoreline_distribution,
    scoring_rate_by_state,
    team_late_goals,
    timeline_exclusions,
    timeline_resilience,
    timeline_team_summary,
    top_scorelines,
    zero_zero_at_60,
)
from ligat.viz.match_flow import chapter_d_chart

exclusions_d = timeline_exclusions(tables)
display(exclusions_d)

<div dir="rtl">

### התוצאות על הלוח

**השאלה:** אילו תוצאות שכיחות בליגה?

**מה לחפש בגרף או בטבלה:** המפה היא שיעור מכל 4,315 המשחקים, בלי נרמול מחדש לאחר חיתוך. n_beyond הוא מספר המשחקים שמחוץ ל־9×9.

**התשובה הכדורגלנית:** אין משחקים מעבר ל־8 שערים לצד; כל מסת התוצאות נמצאת במפה. 1:1 היא התוצאה השכיחה. [scores]

</div>

In [ ]:
scorelines_d = scoreline_distribution(tables)
display(
    pd.DataFrame(
        [
            {
                "n_matches": scorelines_d.attrs["n_matches"],
                "n_beyond": scorelines_d.attrs["n_beyond"],
            }
        ]
    )
)

<div dir="rtl">

### גרף D1 — מפת התוצאות

**השאלה:** מהו המרכז של מפת התוצאות?

**מה לחפש בגרף או בטבלה:** בית בציר האנכי וחוץ באופקי; המספר בכל תא הוא אחוז המשחקים.

**התשובה הכדורגלנית:** 1:1 נצפה ב־523 משחקים, 12.1% מהמדגם; 1:0 ביתי ב־10.5%. [scores]

</div>

In [ ]:
chapter_d_chart("scorelines", scorelines_d).show()

<div dir="rtl">

### עשר התוצאות השכיחות

**השאלה:** האם תוצאה אחת מספיקה כדי לתאר את הליגה?

**מה לחפש בגרף או בטבלה:** n הוא מספר ההופעות; rate ורווח 95% מתוך כל המשחקים.

**התשובה הכדורגלנית:** גם התוצאה השכיחה, 1:1, מתארת רק כ־12% מהמשחקים — צריך התפלגות ולא ניחוש בודד. [scores]

</div>

In [ ]:
top_scores_d = top_scorelines(tables)
display(top_scores_d.round(4))

<div dir="rtl">

### נקודת ההשוואה של Poisson

**השאלה:** איך תיראה הליגה אם שערי הצדדים בלתי תלויים ובעלי ממוצעים קבועים?

**מה לחפש בגרף או בטבלה:** lambda_home/lambda_away הם ממוצעי כל משחקי הפיתוח; הזנב נשאר מחוץ למפה ומדווח, לא מנורמל פנימה.

**התשובה הכדורגלנית:** ממוצע הבית 1.324 שערים והחוץ 1.140; מסת הזנב מעבר למפה כ־0.00138%. זו השוואה תיאורית בלבד. [scores]

</div>

In [ ]:
poisson_d = poisson_reference(tables)
display(pd.DataFrame([poisson_d.attrs]))

<div dir="rtl">

### נצפה מול Poisson

**השאלה:** האם יש עודף תיקו או שהפער נמצא בתוצאות מסוימות?

**מה לחפש בגרף או בטבלה:** ratio גדול מ־1 הוא עודף יחסי; all_draws כולל גם את זנב התיקו מחוץ למפה. רווחי הסמך הם לנצפה, לא אי־ודאות של תחזית.

**התשובה הכדורגלנית:** תיקו כולל: 27.115% נצפה מול 27.089% בנקודת ההשוואה; 0:0 שכיח בכ־11.7% יחסיים יותר, אך 1:1 בכ־5.6% פחות. [scores]

</div>

In [ ]:
poisson_comparison_d = observed_vs_poisson(tables)
display(poisson_comparison_d.round(4))

<div dir="rtl">

### גרף D2 — כל התיקו מול התוצאות הנמוכות

**השאלה:** האם שיעור התיקו הכולל מצדיק תיקון גורף?

**מה לחפש בגרף או בטבלה:** השוו כל תוצאה בנפרד ואת סך התיקו; הבדלי חוזק אינם מנוטרלים כאן.

**התשובה הכדורגלנית:** יש רק כ־0.094% יותר תיקו יחסית, כמעט ללא פער כולל. במחברת 06 נשקול Dixon-Coles לפי מבנה התוצאות הנמוכות וביצועים מחוץ למדגם, ולא בגלל טענה לעודף תיקו כללי. [scores]

</div>

In [ ]:
chapter_d_chart("poisson", poisson_comparison_d).show()

<div dir="rtl">

### סוגי התיקו לאורך העונות

**השאלה:** האם כל תיקו הוא אותו סוג משחק?

**מה לחפש בגרף או בטבלה:** 0:0, 1:1 ו־2:2+ מחושבים מתוך כל המשחקים בכל עונה; סכומם הוא שיעור התיקו באותה עונה.

**התשובה הכדורגלנית:** בסך הכול 410 משחקי 0:0 לעומת 523 משחקי 1:1; הפירוק העונתי מאפשר לבדוק אם ממוצע הליגה מסתיר תקופות שונות. [scores]

</div>

In [ ]:
draw_types_d = draw_types(tables)
display(draw_types_d.round(4))

<div dir="rtl">

### האם השערים מפוזרים כמו Poisson?

**השאלה:** האם השונות שווה לממוצע?

**מה לחפש בגרף או בטבלה:** מעל 1 פיזור־יתר; שלוש השורות מפרידות בית, חוץ וסך הכול. רווחי הסמך דוגמים עונות שלמות ושומרים צמדי תוצאות יחד.

**התשובה הכדורגלנית:** היחסים כ־1.089 בבית ו־1.082 בחוץ, מול 0.998 בסך הכול; פיזור הסכום יכול להסתיר תלות בין הצדדים. [scores]

</div>

In [ ]:
dispersion_d = goal_count_dispersion(tables)
display(dispersion_d.round(4))

<div dir="rtl">

### גרף D3 — פיזור ותלות

**השאלה:** למה אי אפשר להסיק מעמודת סך השערים בלבד?

**מה לחפש בגרף או בטבלה:** הקו בגובה 1 הוא נקודת Poisson; בכותרת המתאם ורווח 95%.

**התשובה הכדורגלנית:** מתאם בית–חוץ הוא כ־−0.081; חלק גדול ממנו צפוי מהבדלי חוזק — קבוצה חזקה מבקיעה יותר ויריבתה פחות. התלות בתוך המשחק תיבדק רק אחרי ש-goals יקבע λ לכל צד, במחברת 06. אין להסיק מהמתאם הכולל לבדו אם השערים תלויים לאחר התחשבות בכוח. [scores]

</div>

In [ ]:
chapter_d_chart("dispersion", dispersion_d).show()

<div dir="rtl">

### מה השתנה לאורך המשחק?

**השאלה:** כמה זמן באמת הייתה קבוצה ביתרון?

**מה לחפש בגרף או בטבלה:** הדוגמה מעונת 2023/24 (EXAMPLE_SEASON), עם שתי שורות לכל משחק; דקות של יתרון, שוויון ופיגור מסתכמות ל־90. ערכי NA במדדי הנקודות מציינים שלא הייתה חשיפה למצב.

**התשובה הכדורגלנית:** יש 8,602 שורות תקינות אחרי 14 החרגות. במשחק 0:0 כל 90 הדקות הן בשוויון, בלי יתרון או פיגור מלאכותיים. [goals, motivation]

</div>

In [ ]:
timelines_d = game_timelines(tables)
display(timelines_d.loc[timelines_d.season_id.eq(EXAMPLE_SEASON)].head(8))

<div dir="rtl">

### מחצית אל הסיום

**השאלה:** עד כמה יתרון במחצית נשמר?

**מה לחפש בגרף או בטבלה:** שיעורי W/D/L בכל שורת מצב מחצית מסתכמים ל־1; הפירוק לבית וחוץ משאיר את הביתיות גלויה.

**התשובה הכדורגלנית:** המובילה במחצית מנצחת ב־77.3% בבית וב־72.4% בחוץ; נשאר סיכוי ממשי לשינוי. מוחרגות 28 שורות קבוצה×משחק. [הבנת המשחק]

</div>

In [ ]:
halftime_d = halftime_transitions(tables)
display(halftime_d.round(4))

<div dir="rtl">

### גרף D4 — מחצית מול סיום

**השאלה:** האם יתרון מחצית בבית ובחוץ דומה?

**מה לחפש בגרף או בטבלה:** כל שורה היא התפלגות נפרדת; n ורווחים בטבלה שמעל.

**התשובה הכדורגלנית:** יתרון במחצית מסתיים בהפסד בכ־6.2% בבית ובכ־8.8% בחוץ; יתרון אינו הבטחה. [הבנת המשחק]

</div>

In [ ]:
chapter_d_chart("halftime", halftime_d).show()

<div dir="rtl">

### 0:0 אחרי שעה

**השאלה:** אם לא נכבש שער עד דקה 60 כולל, האם המשחק סגור?

**מה לחפש בגרף או בטבלה:** הגדרה לפי השעון 0–90. הוספנו scoring_draw כדי שגם 1:1 מאוחר ודומיו יהיו בחלוקה, ולא ייעלמו מהמכנה.

**התשובה הכדורגלנית:** מתוך 1,003 משחקים כאלה, 40.9% הסתיימו 0:0, 31.0% בניצחון בית, 21.4% בניצחון חוץ ו־6.7% בתיקו עם שערים. [הבנת המשחק]

</div>

In [ ]:
zero60_d = zero_zero_at_60(tables)
display(zero60_d.round(4))

<div dir="rtl">

### מי שכבשה ראשונה

**השאלה:** כמה נקודות בדרך כלל נשמרות אחרי שער ראשון?

**מה לחפש בגרף או בטבלה:** 0:0 וזהות ראשונה עמומה אינם מסומנים כמי שספגה ראשונה. אותה דקה בשני הצדדים נשארת עמומה בניתוח הזה; בציר הזמן משתמשים בסדר goal_number המוסכם.

**התשובה הכדורגלנית:** המבקיעה ראשונה בבית מנצחת בכ־72.5% וצוברת 2.35 נקודות; בחוץ כ־65.7% ו־2.19. 3,890 משחקים עם זהות ראשונה ידועה נכנסים להשוואה. [הבנת המשחק]

</div>

In [ ]:
first_goal_d = first_goal_effect(tables)
display(first_goal_d.round(4))

<div dir="rtl">

### גרף D5 — יתרון של השער הראשון

**השאלה:** האם הביתיות נעלמת אחרי פתיחת התוצאה?

**מה לחפש בגרף או בטבלה:** רווחי הסמך והמכנים שונים בין הקבוצות; ניצחון, תיקו והפסד מוצגים בנפרד.

**התשובה הכדורגלנית:** גם אחרי שער ראשון נשאר פער של כ־6.9 נקודות אחוז בסיכוי הניצחון בין בית לחוץ. זו אינה ההשפעה הסיבתית של שער. [הבנת המשחק]

</div>

In [ ]:
chapter_d_chart("first_goal", first_goal_d).show()

<div dir="rtl">

### מתי נפתח הסכר?

**השאלה:** באילו דקות נכבש השער הראשון?

**מה לחפש בגרף או בטבלה:** כל דלי הוא שיעור מהמשחקים התקינים; no_goal הוא 0:0 בנפרד. 45+ אינו מתערבב עם תחילת המחצית השנייה.

**התשובה הכדורגלנית:** 410 משחקים תקינים נשארו 0:0; שערי מחצית ראשונה בדקה 46+ שייכים לתוספת הראשונה, לא למחצית השנייה. [הבנת המשחק]

</div>

In [ ]:
first_minute_d = first_goal_minute_distribution(tables)
display(first_minute_d.round(4))

<div dir="rtl">

### טבלת עזר — מתי מובקעים שערים ובאיזה מצב?

**השאלה:** האם שער מאוחר נכבש בדרך כלל ביתרון או בניסיון לחזור?

**מה לחפש בגרף או בטבלה:** n הוא מספר כל השערים בדלי, count השערים במצב המסוים; שלושת השיעורים מסתכמים ל־1 בכל דלי לא־ריק. סדר אותה דקה תיאורי בלבד.

**התשובה הכדורגלנית:** ברבע השעה הראשון 1,188 מ־1,347 השערים, 88.2%, הובקעו כשהמבקיעה בשוויון. זו תדירות שערים, לא הסתברות להבקיע בכל דקת חשיפה. [goals, motivation]

</div>

In [ ]:
timing_d = goal_timing(tables)
display(timing_d.round(4))

<div dir="rtl">

### קצב הבקעה לאחר נרמול לזמן החשיפה

**השאלה:** כמה שערים נכבשים לכל 90 דקות שבהן הקבוצה מובילה, בשוויון או מפגרת?

**מה לחפש בגרף או בטבלה:** n_goals, minutes וקצב goals_per_90 עם רווח 95% מדגימת עונות. ההחרגות זהות לציר הזמן; שערים עם score_before חסר מדווחים בנפרד, בלי לנחש את מצבם. דקות החשיפה נשארות במכנה. אין כאן התאמה לחוזק הקבוצות.

**התשובה הכדורגלנית:** בפיגור הקצב גבוה מבשוויון גם בבית (1.29 מול 1.19) וגם בחוץ (1.17 מול 1.03). נתון של 88% משערי רבע השעה הראשון בשוויון נשאר בטבלת העזר, אבל אינו עונה לשאלה על קצב הבקעה. [goals, motivation]

</div>

In [ ]:
scoring_rates_d = scoring_rate_by_state(tables)
display(scoring_rates_d.round(4))
display(
    pd.DataFrame(
        [
            {
                "n_missing_score_before": scoring_rates_d.attrs["n_missing_score_before"],
                "n_excluded_matches": scoring_rates_d.attrs["n_excluded_matches"],
            }
        ]
    )
)

<div dir="rtl">

### גרף D6 — קצב הבקעה לפי מצב התוצאה

**השאלה:** האם מפגרת מבקיעה מהר יותר מאשר בשוויון, לאחר חלוקה בזמן החשיפה?

**מה לחפש בגרף או בטבלה:** בית וחוץ בנפרד; שערים ל־90 דקות מתוך הזמן שהקבוצה נמצאה במצב. רווחי סמך מדגימת עונות שלמות, ומכני השערים והדקות בריחוף. לא משווים רק את מספר השערים.

**התשובה הכדורגלנית:** בפיגור מול שוויון: בית 1.29 מול 1.19, חוץ 1.17 מול 1.03 שערים ל־90 דקות. מפגרת מבקיעה מהר יותר מאשר בשוויון — דפוס שמתאים ללחץ לחזור, לא הוכחה לסיבה. המובילה בקצב הגבוה ביותר (1.62 בבית, 1.35 בחוץ) בעיקר משום שקבוצות חזקות מובילות יותר: הרכב הקבוצות במצב, לא השפעה סיבתית של יתרון. [goals, motivation]

</div>

In [ ]:
chapter_d_chart("timing", scoring_rates_d).show()

<div dir="rtl">

### פרופיל זמן לקבוצה ולעונה

**השאלה:** איך מתחבר זמן ביתרון לנקודות?

**מה לחפש בגרף או בטבלה:** ממוצעי הדקות מדויקים לעונה המתוארת; לשוויון בתוך 10 דקות n נפרד ורווח Wilson. לא מסיקים מקבוצת חשיפה קטנה מ־30.

**התשובה הכדורגלנית:** הדוגמה מ־2023/24 מציגה פרופילים מתוך 14 הקבוצות באותה עונה. בכלל 252 קבוצות־העונות, זמן ההובלה הממוצע נע בין כ־3.2 ל־48.5 דקות; אלה הבדלים גדולים במה שנראה באותן 90 דקות, לאחר המשחקים. [goals, motivation]

</div>

In [ ]:
timeline_summary_d = timeline_team_summary(tables)
display(timeline_summary_d.xs(EXAMPLE_SEASON, level="season_id").head(8).round(3))

<div dir="rtl">

### גרף D7 — זמן ביתרון ונקודות

**השאלה:** האם יש קבוצות שמחזיקות יתרון בלי לצבור הרבה נקודות?

**מה לחפש בגרף או בטבלה:** נקודה אחת לקבוצה×עונה, צבע לפי עונה וריחוף לשם ולמכנה. כל המידע מגיע לאחר המשחקים.

**התשובה הכדורגלנית:** ב־252 קבוצות־עונות זמן ההובלה נע בין כ־3.2 ל־48.5 דקות למשחק. רואים יחד את הזמן ואת הניקוד שנוצרו באותם משחקים; זה תיאור בדיעבד ואינו מוכיח שמשך היתרון לבדו גרם לנקודות. דוגמת הטבלה היא 2023/24. [goals, motivation]

</div>

In [ ]:
chapter_d_chart("time_in_state", timeline_summary_d).show()

<div dir="rtl">

### חוסן מצטבר

**השאלה:** מי אוספת נקודות אחרי שפיגרה ומי מאבדת אחרי שהובילה?

**מה לחפש בגרף או בטבלה:** מועדונים עם לפחות 5 עונות; n_led/n_trailed הם מספר המשחקים שבהם הייתה חשיפה. השיעורים מחלקים את הנקודות במספר משחקי הפיגור או היתרון; הסכומים נשמרים לצדם. רווחי הסמך דוגמים עונות שלמות, וכל מועדון מיוצג רק אם יש לפחות חמש עונות.

**התשובה הכדורגלנית:** הפועל ב״ש משיגה כ־0.80 נקודות לכל משחק שבו פיגרה ומאבדת כ־0.44 לכל משחק שבו הובילה; ת״א 0.79/0.40, חיפה 0.77/0.41 והפועל ת״א 0.53/0.72. הנרמול לחשיפה מאפשר השוואה שימושית יותר מסכומים, אך חוזק היריבות ומצב המשחק אינם מנוטרלים. [goals, motivation]

</div>

In [ ]:
resilience_d = timeline_resilience(tables)
display(resilience_d)

<div dir="rtl">

### גרף D8 — נקודות מיתרון ומפיגור

**השאלה:** איך נראים שני צדי החוסן יחד?

**מה לחפש בגרף או בטבלה:** הצירים הם נקודות לכל משחק שבו הייתה חשיפה; רווחים בשני הצירים ומכני n_led/n_trailed בריחוף. גודל הנקודה הוא מספר המשחקים; שתי הקבוצות יכולות להופיע בשני המדדים באותו משחק. המשחקים שכבר תוארו אינם מדגם לניבוי.

**התשובה הכדורגלנית:** ב״ש ות״א משיגות כ־0.80 ו־0.79 נקודות לכל משחק בפיגור, מול כ־0.53 להפועל ת״א. איבוד מיתרון הוא כ־0.44, 0.40 ו־0.72 בהתאמה. אלה קשרים תיאוריים עם רווחי אי־ודאות, לא הוכחה לתכונת אופי או להשפעה סיבתית. [goals, motivation]

</div>

In [ ]:
chapter_d_chart("resilience", resilience_d).show()

<div dir="rtl">

### קבוצות של סוף משחק

**השאלה:** מי מבקיעה חלק גדול משעריה מאוחר?

**מה לחפש בגרף או בטבלה:** רק מועדונים עם לפחות 5 עונות; second_half_goal_share כולל את כל המחצית השנייה, late_goal_share רק 76+ כולל 90+. n_goals הוא המכנה.

**התשובה הכדורגלנית:** למכבי ת״א 24.5% משעריה מדקה 76, מול 22.1% לחיפה; הרווחים חופפים, ולכן אין כאן הכרעה על יתרון של מועדון אחד. [goals]

</div>

In [ ]:
late_goals_d = team_late_goals(tables)
display(late_goals_d.round(4))

<div dir="rtl">

### גרף D9 — שערים מדקה 76

**השאלה:** האם סדר העמודות הוא דירוג יציב של יכולת מאוחרת?

**מה לחפש בגרף או בטבלה:** השוו רווחים ולא רק גובה; שיעור מתוך שערים אינו שיעור מתוך משחקים או דקות.

**התשובה הכדורגלנית:** הפער התיאורי ת״א–חיפה הוא כ־2.4 נקודות אחוז, אבל טווחי אי־הוודאות חופפים. נדרש מחקר נוסף לפני הפיכתו לפיצ׳ר. [goals]

</div>

In [ ]:
chapter_d_chart("late_goals", late_goals_d).show()

<div dir="rtl">

### פנדלים ועצמיים לקבוצה

**השאלה:** איזה חלק מהשערים נזקף מפנדל או מרגל יריבה?

**מה לחפש בגרף או בטבלה:** שער עצמי לטובת קבוצה הוא שער שנזקף לזכותה, לא לזהות המבקיע. רק המשחק ללא מבקיעים אמינים מוחרג; סתירת מחצית אינה פוגעת בספירת סוגי שערים.

**התשובה הכדורגלנית:** שערי הפנדל למשחק עלו מכ־0.16 בעונה הראשונה (2006/07) לכ־0.25 באחרונה (2023/24). זו מגמה בשערים שהובקעו, לא בהכרח במספר השריקות או באחוז ההצלחה — החמצות אינן בדאטה. בטבלה per_match הוא המכנה להשוואה זו; גרף D10 מציג בנפרד את החלק מכל השערים. [goals, scores]

</div>

In [ ]:
penalties_teams_d = penalty_owngoal_stats(tables)
display(penalties_teams_d.head(8).round(4))

<div dir="rtl">

### סיכום שערי פנדל ועצמיים בליגה

**השאלה:** האם צריך לספור כל שער פעמיים, לטובה ולחובה?

**מה לחפש בגרף או בטבלה:** בסיכום הליגה כל שער נספר פעם אחת. per_match יכול להיות גדול מ־1 ואינו שיעור בינומי; רווחי Wilson מתייחסים לחלק מכל השערים.

**התשובה הכדורגלנית:** שערי הפנדל למשחק עלו מכ־0.16 בעונה הראשונה (2006/07) לכ־0.25 באחרונה (2023/24). זו מגמה בשערים שהובקעו, לא בהכרח במספר השריקות או באחוז ההצלחה — החמצות אינן בדאטה. בטבלה per_match הוא המכנה להשוואה זו; גרף D10 מציג בנפרד את החלק מכל השערים. [goals, scores]

</div>

In [ ]:
penalties_seasons_d = penalty_owngoal_by_season(tables)
display(penalties_seasons_d.round(4))

<div dir="rtl">

### גרף D10 — שערי פנדל ועצמיים לאורך העונות

**השאלה:** האם שיעורם משתנה מעונה לעונה?

**מה לחפש בגרף או בטבלה:** לכל עונה חלק מתוך השערים ומכנה n_goals; רווחים חופפים הם סיבה לזהירות. אין הסקת סיבה על שופטים.

**התשובה הכדורגלנית:** שערי הפנדל למשחק עלו מכ־0.16 בעונה הראשונה (2006/07) לכ־0.25 באחרונה (2023/24). זו מגמה בשערים שהובקעו, לא בהכרח במספר השריקות או באחוז ההצלחה — החמצות אינן בדאטה. בטבלה per_match הוא המכנה להשוואה זו; גרף D10 מציג בנפרד את החלק מכל השערים. [goals, scores]

</div>

In [ ]:
chapter_d_chart("penalties", penalties_seasons_d).show()

<div dir="rtl">

### טעינת אירועי אדומים מוגבלת

**השאלה:** מה נוסף מעבר למספר האדומים במשחק?

**מה לחפש בגרף או בטבלה:** הקריאה מסוננת לפי מזהי משחקי הפיתוח שכבר נטענו. אין קריאה לעונות החזקה או שינוי בנתונים.

**התשובה הכדורגלנית:** 1,167 אדומים זמינים במשחקי הפיתוח; ב־12 חסרה הדקה. הם יישמרו בספירה ובצהוב שני. [הבנת המשחק]

</div>

In [ ]:
red_events_d = Repository().read_filtered(
    "red_cards", [("game_id", "in", tables.match_table.index.tolist())]
)
display(red_events_d.head(8))

<div dir="rtl">

### אדום לקבוצה ואדום ליריבה

**השאלה:** מה הקשר בין מאזן האדומים לתוצאה?

**מה לחפש בגרף או בטבלה:** קבוצת 1:0 היא אדום לקבוצה ולא ליריבה; 0:1 המצב ההפוך. סכום n בכל ארבע הקבוצות הוא 8,630 שורות קבוצה×משחק.

**התשובה הכדורגלנית:** קבוצה שרק היא קיבלה אדום ניצחה ב־15.8% מהמשחקים האלה, מול 58.8% כשקיבלה רק היריבה. קשר, לא סיבה — לא ידוע כאן מה היה המצב לפני ההרחקה. [הבנת המשחק]

</div>

In [ ]:
red_stats_d = red_card_stats(tables, red_cards=red_events_d)
display(red_stats_d.xs("cards", level="part").round(4))

<div dir="rtl">

### מועד ההרחקה

**השאלה:** האם אדום מוקדם ומאוחר מספרים אותו סיפור?

**מה לחפש בגרף או בטבלה:** אם חסרה דקה לאחד מאדומי הקבוצה, איננו יודעים איזה היה הראשון; כל שורת הקבוצה הזאת מוחרגת מפירוק הזמן, בלבד.

**התשובה הכדורגלנית:** 432 שורות קבוצה×משחק עם אדום ראשון עד 60 ו־641 אחריה; 12 שורות ללא דקה אינן משויכות למועד מומצא. גם כאן מצב המשחק עלול להסביר חלק מהקשר. [הבנת המשחק]

</div>

In [ ]:
display(red_stats_d.xs("first_card_minute", level="part").round(4))

<div dir="rtl">

### צהוב שני מתוך האדומים

**השאלה:** כמה מההרחקות הגיעו מצהוב שני?

**מה לחפש בגרף או בטבלה:** n הוא מספר האדומים בעונה, לא מספר המשחקים; עמודות תוצאות שאינן חלות על החלק הזה אינן מוצגות.

**התשובה הכדורגלנית:** ב־2006/07 נרשמו 30 הרחקות בצהוב שני מתוך 76 אדומים, 39.5%; אין מכאן מידע על כלל הצהובים. [הבנת המשחק]

</div>

In [ ]:
display(
    red_stats_d.xs("second_yellow", level="part")[
        [
            "n",
            "second_yellow_n",
            "second_yellow_share",
            "second_yellow_share_ci_lower",
            "second_yellow_share_ci_upper",
        ]
    ].round(4)
)

<div dir="rtl">

### גרף D11 — אדומים ותוצאה

**השאלה:** האם ההרחקה היא הסיבה להפסד?

**מה לחפש בגרף או בטבלה:** לא ניתן להסיק זאת כאן: הקבוצות שונות בכוח, במצב ובדקות החשיפה. רווחי הסמך אינם מתקנים את הערבוב הזה.

**התשובה הכדורגלנית:** 15.8% ניצחונות כשאדום רק לקבוצה מול 58.8% כשאדום רק ליריבה הוא קשר חזק בתיאור, לא אפקט סיבתי של כרטיס. [הבנת המשחק]

</div>

In [ ]:
chapter_d_chart("red_cards", red_stats_d).show()

<div dir="rtl">

## מה למדנו — פעימה D

1. **תוצאות ותיקו:** 1:1 היא השכיחה (523 משחקים, 12.1%). סך התיקו **27.115% מול 27.089%** בנקודת Poisson העצמאית — פער זעיר כולל. ב־0:0 יש עודף יחסי **11.7%**, וב־1:1 חוסר **5.6%**. השוואת Dixon-Coles במחברת 06 תתבסס על פירוק וביצועים, לא על עודף כללי שלא ראינו. [scores]
2. **פיזור ותלות:** שונות/ממוצע **1.089 בבית**, **1.082 בחוץ**, **0.998 בסכום**; מתאם הצדדים **−0.081**. חלק גדול מהמתאם השלילי צפוי מהבדלי חוזק: חזקה מבקיעה יותר ויריבתה פחות. התלות בתוך המשחק תיבדק רק אחרי ש-goals יקבע λ לכל צד, במחברת 06; המתאם הכולל אינו בדיקת תלות שיורית. [scores]
3. **מחצית והשער הראשון:** יתרון מחצית נשמר לניצחון ב־**77.3% בבית ו־72.4% בחוץ**; שער ראשון ב־**72.5% ו־65.7%**. מתוך 1,003 משחקי 0:0 אחרי שעה, **40.9%** נשארו 0:0 — רובם בכל זאת קיבלו שער. [הבנת המשחק]
4. **זמן במצב וחוסן:** 4,301 משחקים יוצרים **8,602 שורות**, ובכל אחת 90 דקות. לאחר נרמול לחשיפה, ב״ש משיגה כ־**0.80** נקודות למשחק עם פיגור ומאבדת **0.44** למשחק עם יתרון; ת״א **0.79/0.40**, חיפה **0.77/0.41**, הפועל ת״א **0.53/0.72**. החישוב מתייחס רק למשחקים עם המצב המתאים, עם רווחים מדגימת עונות; קשר, לא תכונת אופי שהוכחה. [goals, motivation]
5. **סיום המשחק:** חלק השערים מדקה 76 הוא **24.5% בת״א מול 22.1% בחיפה**, ברווחים חופפים. לפי דקות החשיפה, מפגרת מבקיעה מהר יותר מאשר בשוויון: **1.29 מול 1.19** בבית, **1.17 מול 1.03** בחוץ — דפוס שמתאים ללחץ לחזור, לא הוכחת סיבה. המובילה גבוהה עוד יותר (**1.62/1.35**), בעיקר מפני שקבוצות חזקות מובילות יותר; הרכב, לא סיבה. [goals, motivation]
6. **אדומים:** ניצחון בכ־**15.8%** כשאדום רק לקבוצה לעומת **58.8%** כשרק ליריבה; **12** כרטיסים ללא דקה נשמרים בספירות. זה קשר, לא סיבה. שערי פנדל למשחק עלו מכ־**0.16 ל־0.25** בין העונה הראשונה לאחרונה; הם מתארים רק בעיטות שהובקעו, ללא שריקות או החמצות. [הבנת המשחק, goals]

### מה מגיע עכשיו?
פעימה D נעצרת לסקירת Claude Code ועמית. פעימה E תתחיל רק אחרי QA, אישור מיזוג ובדיקת יציבות.

</div>